# Maximum likelihood estimation
The five mouse weights and the five call counts of the Note: grid search, log-likelihood, the derivative recipe,
numerical minimisation of the negative log-likelihood, and a simulation of how the MLE improves with more data.

## 0. One mouse, two mice, and the bag of balls
One mouse at 32 g under N(mu, 2^2): the likelihood is the height of the curve. Two mice (32, 34): multiply. One point cannot give sigma. Five green balls in five draws: the likelihood p^5 is largest at p = 1.

In [ ]:
import numpy as np
from scipy import stats
f = lambda x, m, s=2: stats.norm(m, s).pdf(x)
print("one mouse at 32:", {m: round(f(32, m), 3) for m in (28, 30, 32)})
print("two mice, mu = 28:", f"{f(32, 28) * f(34, 28):.2e}", " heights", round(f(32, 28), 4), round(f(34, 28), 4))
print("two mice, mu = 33:", round(f(32, 33) * f(34, 33), 3))
for s in (2, 0.5, 0.1, 0.01):
    print(f"sigma = {s}: likelihood of one mouse at its own mean = {f(32, 32, s):.2f}")
print("bag: p = 2/5 ->", round(0.4**5, 3), "  p = 4/5 ->", round(0.8**5, 3), "  p = 1 ->", 1.0**5)

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats, optimize

np.set_printoptions(precision=4, suppress=True)
# one seeded random generator for the whole notebook
rng = np.random.default_rng(0)
mice = np.array([29, 31, 32, 33, 35.0])
calls = np.array([2, 1, 3, 2, 2])

## 1. The likelihood of one curve: product of five heights

In [ ]:
# heights of N(28, 2^2) above each mouse
heights = stats.norm(28, 2).pdf(mice)
print(heights)
print("likelihood:", heights.prod())

## 2. Grid search for the mean (sigma fixed at 2)

In [ ]:
mus = np.arange(26, 38.001, 0.01)
# likelihood of every mean on the grid
L = np.array([stats.norm(m, 2).pdf(mice).prod() for m in mus])
print("best mean on the grid:", mus[L.argmax()].round(2), " average of the data:", mice.mean())
fig = go.Figure(go.Scatter(x=mus, y=L, mode="lines"))
fig.update_layout(template="simple_white", xaxis_title="mean", yaxis_title="likelihood", height=350)
fig.show()

## 3. Both parameters at once: the log-likelihood surface

In [ ]:
m_grid = np.linspace(29, 35, 121)
s_grid = np.linspace(0.8, 5, 211)
M, S = np.meshgrid(m_grid, s_grid)
# logpdf avoids the tiny products; sum over the five mice
LL = stats.norm(M[..., None], S[..., None]).logpdf(mice).sum(axis=-1)
i, j = np.unravel_index(LL.argmax(), LL.shape)
print("grid MLE: mu =", M[i, j].round(2), " sigma =", S[i, j].round(2))
# the closed form: mean, and standard deviation dividing by n
print("closed form:", mice.mean(), mice.std())
fig = go.Figure(go.Contour(x=m_grid, y=s_grid, z=LL, contours=dict(start=-25, end=-10.5, size=0.5)))
fig.update_layout(template="simple_white", xaxis_title="mu", yaxis_title="sigma", height=450)
fig.show()

## 4. Underflow: why we sum logs

In [ ]:
# 2000 mice from the same population
big = rng.normal(32, 2, size=2000)
print("product of densities:", stats.norm(32, 2).pdf(big).prod())
print("sum of log densities:", stats.norm(32, 2).logpdf(big).sum())

## 5. The derivative recipe for the Poisson rate

In [ ]:
n, total = len(calls), calls.sum()
# derivative of the log-likelihood: total / lam - n
slope = lambda lam: total / lam - n
for lam in (1, 2, 3.5):
    print(f"lambda = {lam}: slope = {slope(lam):+.2f}")
print("MLE from the formula:", total / n)

## 6. The same answer by minimising the NLL numerically

In [ ]:
nll = lambda lam: -stats.poisson(lam).logpmf(calls).sum()
res = optimize.minimize_scalar(nll, bounds=(0.01, 10), method="bounded")
print("numerical MLE:", round(res.x, 4), " NLL there:", round(res.fun, 3))
# two parameters: minimise the normal NLL over (mu, log sigma) so sigma stays positive
nll2 = lambda p: -stats.norm(p[0], np.exp(p[1])).logpdf(mice).sum()
res2 = optimize.minimize(nll2, x0=[30, 0])
print("normal MLE: mu =", res2.x[0].round(4), " sigma =", np.exp(res2.x[1]).round(4))

## 7. Consistency: the MLE homes in on the truth as n grows
Draw many samples of size n from Po(2), compute the MLE (the mean) of each, and look at the spread.

In [ ]:
for n in (5, 20, 80, 320):
    est = rng.poisson(2, size=(5000, n)).mean(axis=1)
    print(f"n = {n:4d}: average MLE {est.mean():.3f}, spread (std) {est.std():.3f}")

Each time n is multiplied by 4, the spread halves: the variance falls like 1/n.